# LLM & RAG Pipeline — Sales Chatbot (single-notebook build)

Self-contained version: retrieval engine (P2), LLM manager, prompt builder, memory, human-handoff policy, and the RAG orchestrator all live in this one notebook, and the retrieval index is built from the **real `products_clean.csv` catalogue** (913 products) instead of the 4-item mock catalogue. This makes the notebook easy to deploy as-is (e.g. copy into a Gradio `app.py` on Hugging Face Spaces) without needing sibling `.py` files.

## 0. Setup

In [14]:
%pip install -q faiss-cpu sentence-transformers rank-bm25 pandas \
                 torch transformers \
                 langchain-openai langchain-core \
                 requests python-dotenv

In [15]:
import os
import sys
import json
from typing import Any, Callable, Dict, List, Optional, Tuple

from dotenv import load_dotenv

# Loads OPENROUTER_API_KEY / GEMINI_API_KEY / ADMIN_PASSWORD from a .env file
# in the project root, per the README's "Setup & Installation" step 3.
load_dotenv()

# Make sure any sibling modules copied next to this notebook are importable.
sys.path.append(os.getcwd())


def get_secret(name: str) -> Optional[str]:
    """Read a secret from Colab userdata if running in Colab, else from the environment.

    This keeps the notebook portable: it works unchanged in Colab (using the
    Secrets panel), locally with a .env file, or on a deployment host (e.g. HF
    Spaces) using regular environment variables / repo secrets.
    """
    try:
        from google.colab import userdata  # type: ignore
        try:
            value = userdata.get(name)
            if value:
                return value
        except Exception:
            pass
    except ImportError:
        pass
    return os.getenv(name)


print("OPENROUTER_API_KEY set:", bool(get_secret("OPEN_ROUTER_KEY")))
print("GEMINI_API_KEY set:", bool(get_secret("GEMINI_API_KEY")))

OPENROUTER_API_KEY set: True
GEMINI_API_KEY set: True


## 1. Product Catalogue — load & clean `products_clean.csv`

Loads the real 913-product catalogue directly into memory with pandas (no SQLite step). Keeps only active listings, coerces the retrieval-critical text/numeric fields, and returns a plain list of dicts shaped exactly like what `SalesRetrievalEngine.build_indexes()` expects — the same shape `ProductDatabase.get_all_active_products()` used to produce.

In [16]:
import pandas as pd

# Common locations to look for the CSV so this cell works whether the file
# sits next to the notebook, inside a data/ folder, or (in Colab) hasn't been
# uploaded yet.
DEFAULT_CSV_CANDIDATES = [
    "products_clean.csv",
    "data/products_clean.csv",
    "/content/products_clean.csv",
]

TEXT_COLUMNS = [
    "title", "category", "product_description", "product_specifications",
    "what_customers_said", "currency", "seller_name",
]
NUMERIC_COLUMNS = ["final_price", "initial_price", "discount", "rating", "ratings_count"]


def _find_csv_path(explicit_path: Optional[str] = None) -> str:
    """Locate products_clean.csv, prompting a Colab upload as a last resort."""
    if explicit_path and os.path.exists(explicit_path):
        return explicit_path
    for candidate in DEFAULT_CSV_CANDIDATES:
        if os.path.exists(candidate):
            return candidate
    try:
        from google.colab import files  # type: ignore
        print("products_clean.csv not found locally \u2014 please upload it.")
        uploaded = files.upload()
        for name in uploaded:
            if name.lower().endswith(".csv"):
                return name
    except ImportError:
        pass
    raise FileNotFoundError(
        "products_clean.csv not found. Place it next to this notebook, in a "
        "'data/' subfolder, or upload it when prompted in Colab."
    )


def load_products_from_csv(csv_path: Optional[str] = None) -> List[Dict[str, Any]]:
    """Load + lightly clean the product catalogue for indexing.

    Reads products_clean.csv, keeps only active listings (is_active == 1),
    fills missing text fields with "" and missing numeric fields with 0 so
    downstream code (rich-text building, price filters) never chokes on NaN,
    and returns a list of plain dicts \u2014 the same shape the SQLite-backed
    ProductDatabase used to hand to build_indexes().
    """
    path = _find_csv_path(csv_path)
    df = pd.read_csv(path)

    if "is_active" in df.columns:
        df = df[df["is_active"] == 1].copy()

    for col in TEXT_COLUMNS:
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str)

    for col in NUMERIC_COLUMNS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

    df["product_id"] = df["product_id"].astype(str)

    products = df.to_dict(orient="records")
    print(f"Loaded {len(products)} active products from {path!r}")
    return products

## 2. Retrieval Engine (P2) \u2014 Hybrid FAISS + BM25 search

`SalesRetrievalEngine`, inlined unmodified from `p2_retrieval_engine.py` so the whole pipeline lives in one file. Dense (FAISS) + sparse (BM25) search fused with Reciprocal Rank Fusion, plus cross-sell recommendations, instalment calculation, and the LLM-context builder.

In [17]:
from __future__ import annotations

import json
import logging
import os
import pickle
import re
import unicodedata
from typing import Any, Callable, Dict, List, Optional, Tuple

import faiss
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

logger = logging.getLogger(__name__)

# ── Defaults for the on-disk index files (README convention) ────────────
DEFAULT_DB_PATH = "database/chatbot.db"
DEFAULT_FAISS_PATH = "database/faiss.index"
DEFAULT_BM25_PATH = "database/bm25_index.pkl"

# ── RRF constant (score = 1 / (rank + RRF_K)) ────────────────────────────
RRF_K = 60

# ── Instalment interest-rate table: months -> rate ───────────────────────
INSTALMENT_RATES: Dict[int, float] = {3: 0.10, 6: 0.15, 12: 0.20}

# ── Keywords that trigger a human-agent handoff ──────────────────────────
HANDOFF_KEYWORDS: Tuple[str, ...] = (
    "talk to human",
    "agent",
    "representative",
    "مشكلة",
    "خدمة العملاء",
)

# Unicode ranges: Arabic letters (normalized set) and letter hamza forms.
_ARABIC_ALEF = ("\u0627", "\u0623", "\u0625", "\u0622")  # ا أ إ آ
_ARABIC_DIACRITICS = "\u064B\u064C\u064D\u064E\u064F\u0650\u0651\u0652\u0653\u0654\u0655\u0656\u0657\u0658\u0659\u065A\u065B\u065C\u065D\u065E\u065F"
_ARABIC_TATWEEL = "\u0640"

# Tokenizer: keeps hyphenated alphanumeric tokens (SKUs like "G-15") whole,
# and treats contiguous Arabic letters/digits as tokens.
_TOKEN_RE = re.compile(r"[a-z0-9]+(?:-[a-z0-9]+)*|[\u0621-\u064A0-9]+")


class SalesRetrievalEngine:
    """Deterministic hybrid retrieval + business logic engine (English/Arabic)."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2") -> None:
        self.model_name = model_name
        self._model: Optional[SentenceTransformer] = None

        # Dual indexes
        self.faiss_index: Optional[faiss.IndexFlatL2] = None
        self.bm25: Optional[BM25Okapi] = None

        # Corpus storage
        self.products: List[Dict[str, Any]] = []
        self.categories: List[str] = []
        self.rich_texts: List[str] = []
        self._vectors: Optional[np.ndarray] = None
        self._is_built: bool = False

    # ────────────────────────────────────────────────────────────────────
    # Private helpers
    # ────────────────────────────────────────────────────────────────────

    def _get_model(self) -> SentenceTransformer:
        """Lazy-load the sentence-transformer model (cached for repeated calls)."""
        if self._model is None:
            logger.info("Loading embedding model '%s' ...", self.model_name)
            self._model = SentenceTransformer(self.model_name)
        return self._model

    @staticmethod
    def _normalize(text: str) -> str:
        """Normalize text for both English and Arabic.

        - Lowercase (handles Arabic once letters are de-diacriticised).
        - NFKC unicode normalization.
        - Strip Arabic diacritics and tatweel (ـ).
        - Collapse Arabic letter variants onto canonical bases
          (أ/إ/آ -> ا , ة -> ه , ى -> ي).
        - Collapse runs of whitespace to a single space.
        """
        if not text:
            return ""

        text = unicodedata.normalize("NFKC", str(text))
        text = text.lower()

        # strip Arabic diacritics and elongation marks
        text = re.sub(f"[{_ARABIC_DIACRITICS}{_ARABIC_TATWEEL}]", "", text)

        # harmonise Arabic letter variants (single-char replacements)
        text = text.replace("\u0623", "\u0627").replace("\u0625", "\u0627").replace("\u0622", "\u0627")
        text = text.replace("\u0629", "\u0647")  # ة -> ه
        text = text.replace("\u0649", "\u064A")  # ى -> ي

        return re.sub(r"\s+", " ", text).strip()

    @classmethod
    def _tokenize(cls, text: str) -> List[str]:
        """SKU-aware + Arabic-aware tokenization.

        Uses the normalized text so tokens are already diacritic-free,
        lower-cased and letter-harmonized. Hyphenated alnum tokens (G-15)
        are kept as a single token for exact SKU matching.
        """
        return _TOKEN_RE.findall(cls._normalize(text))

    @staticmethod
    def _build_rich_text(product: Dict[str, Any]) -> str:
        """Combine the most semantically useful product fields into one string."""
        parts = [
            str(product.get("title", "")),
            str(product.get("category", "")),
            str(product.get("product_description", "")),
            str(product.get("product_specifications", "")),
        ]
        return " ".join(p for p in parts if p).strip()

    def _require_built(self) -> None:
        if not self._is_built:
            raise RuntimeError(
                "Indexes are not built yet. Call build_indexes() or load_indexes() first."
            )

    def _require_query(self, query: str) -> str:
        if not query or not query.strip():
            raise ValueError("Query must be a non-empty string.")
        return query.strip()

    # ────────────────────────────────────────────────────────────────────
    # A1-A3. Initialisation & index management
    # ────────────────────────────────────────────────────────────────────

    def build_indexes(
        self,
        products: List[Dict[str, Any]],
        faiss_path: str = DEFAULT_FAISS_PATH,
        bm25_path: str = DEFAULT_BM25_PATH,
    ) -> Tuple[str, str]:
        """Preprocess products, embed them and build FAISS + BM25 indexes.

        Args:
            products: list of product dicts (as produced by ProductDatabase).
            faiss_path: where to persist the FAISS index.
            bm25_path: where to persist the pickled BM25 index.

        Returns:
            Tuple of (faiss_path, bm25_path) for the saved artifacts.

        Raises:
            ValueError: if `products` is empty.
        """
        if not products:
            raise ValueError("Cannot build indexes from an empty product list.")

        # Ensure the parent directory for on-disk artifacts exists.
        for path in (faiss_path, bm25_path):
            parent = os.path.dirname(path)
            if parent:
                os.makedirs(parent, exist_ok=True)

        self.products = list(products)
        self.rich_texts = [self._build_rich_text(p) for p in self.products]

        # Full-catalogue category list — deterministic, NOT embedding-based.
        # hybrid_search() only ever returns a top-k shortlist, so aggregate
        # questions like "what categories do you have" must not rely on it;
        # they'd get a different, incomplete answer on every call.
        self.categories = sorted({
            str(p.get("category", "")).strip()
            for p in self.products
            if str(p.get("category", "")).strip()
        })

        # -- Dense: FAISS (IndexFlatL2 -> Euclidean distance) --------------
        model = self._get_model()
        vectors = model.encode(
            self.rich_texts,
            normalize_embeddings=False,
            show_progress_bar=False,
            convert_to_numpy=True,
        )
        self._vectors = np.ascontiguousarray(vectors, dtype=np.float32)

        dim = self._vectors.shape[1]
        self.faiss_index = faiss.IndexFlatL2(dim)
        self.faiss_index.add(self._vectors)
        faiss.write_index(self.faiss_index, faiss_path)
        logger.info("FAISS index saved to %s (%d vectors, dim=%d)", faiss_path, len(self._vectors), dim)

        # -- Sparse: BM25Okapi over normalized tokens -----------------------
        tokenized = [self._tokenize(t) for t in self.rich_texts]
        self.bm25 = BM25Okapi(tokenized)
        with open(bm25_path, "wb") as fh:
            pickle.dump(self.bm25, fh)
        logger.info("BM25 index saved to %s", bm25_path)

        self._is_built = True
        return faiss_path, bm25_path

    def load_indexes(
        self,
        faiss_path: str,
        bm25_path: str,
        products: List[Dict[str, Any]],
    ) -> None:
        """Load pre-computed indexes from disk to save memory & startup time.

        Args:
            faiss_path: path to a previously saved FAISS index.
            bm25_path: path to a previously saved (pickled) BM25 index.
            products: the same corpus used to build those indexes.

        Note:
            Vector embeddings are NOT recomputed here; the FAISS index file
            already contains them. Only the product dicts are restored, so a
            future encode() (e.g. for recommendations) is still available.

        Raises:
            ValueError / RuntimeError: if paths are missing or unreadable.
        """
        if not os.path.exists(faiss_path) or not os.path.exists(bm25_path):
            raise ValueError(f"Index paths missing: {faiss_path!r} / {bm25_path!r}")

        self.faiss_index = faiss.read_index(faiss_path)
        with open(bm25_path, "rb") as fh:
            self.bm25 = pickle.load(fh)

        self.products = list(products)
        self.rich_texts = [self._build_rich_text(p) for p in self.products]

        # Reconstruct vectors for get_recommendations() without an extra
        # forward pass over the whole corpus: read them straight back from
        # the FAISS index (cheap, deterministic).
        self._vectors = np.array(self.faiss_index.reconstruct_n(0, self.faiss_index.ntotal))

        self._is_built = True
        logger.info("Loaded FAISS (%d vectors) + BM25 from disk.", self.faiss_index.ntotal)

    def build_indexes_from_database(
        self,
        db_path: str = DEFAULT_DB_PATH,
        faiss_path: str = DEFAULT_FAISS_PATH,
        bm25_path: str = DEFAULT_BM25_PATH,
        error_on_empty: bool = True,
    ) -> Tuple[str, str]:
        """Bridge to the existing SQLite pipeline: build indexes from the DB.

        Pulls all active products through ``ProductDatabase.get_all_active_products()``
        (from ``core.data_pipeline``) and delegates to :meth:`build_indexes`.

        Args:
            db_path: path to the SQLite database produced by data_pipeline.
            faiss_path: where to persist the FAISS index.
            bm25_path: where to persist the pickled BM25 index.
            error_on_empty: if True, raise when the DB has no active products.

        Returns:
            Tuple of (faiss_path, bm25_path).

        Raises:
            FileNotFoundError: if the database file does not exist.
            RuntimeError: if the DB has no active products and error_on_empty=True.
        """
        if not os.path.exists(db_path):
            raise FileNotFoundError(
                f"Database not found at {db_path!r}. Run core.data_pipeline first."
            )

        from core.data_pipeline import ProductDatabase  # local import to avoid hard coupling

        db = ProductDatabase(db_path)
        try:
            products = db.get_all_active_products()
        finally:
            db.close()

        if not products:
            if error_on_empty:
                raise RuntimeError(
                    f"Database {db_path!r} has no active products to index."
                )
            logger.warning("Database %s has no active products; nothing indexed.", db_path)
            return faiss_path, bm25_path

        logger.info("Loaded %d active products from %s", len(products), db_path)
        return self.build_indexes(products, faiss_path=faiss_path, bm25_path=bm25_path)

    # ────────────────────────────────────────────────────────────────────
    # B4. Core hybrid retrieval (RRF fusion)
    # ────────────────────────────────────────────────────────────────────

    @staticmethod
    def _rrf_add(score_map: Dict[int, float], positions: List[int]) -> None:
        """Accumulate RRF scores for an ordering of corpus positions.

        rank is the 0-indexed position in `positions`; a document appearing
        early in *either* list gets a high score, and appearing in both lists
        stacks its contribution — which is how RRF rewards agreement.
        """
        for rank, pos in enumerate(positions):
            score_map[pos] = score_map.get(pos, 0.0) + 1.0 / (rank + RRF_K)

    def _dense_search_positions(self, query_vec: np.ndarray, top_n: int) -> List[int]:
        """FAISS search -> ordered list of corpus positions (best-first)."""
        assert self.faiss_index is not None
        k = min(top_n, self.faiss_index.ntotal)
        if k <= 0:
            return []
        _d, idx = self.faiss_index.search(query_vec, k)
        return [int(i) for i in idx[0] if i != -1]

    def _sparse_search_positions(self, query_tokens: List[str], top_n: int) -> List[int]:
        """BM25 -> ordered list of corpus positions (best-first)."""
        assert self.bm25 is not None
        scores = np.asarray(self.bm25.get_scores(query_tokens))
        if scores.size == 0:
            return []
        k = min(top_n, scores.size)
        # np.argpartition is O(n) — we only fully sort the top-k slice.
        part = np.argpartition(-scores, k - 1)[:k]
        order = part[np.argsort(-scores[part], kind="stable")]
        return [int(i) for i in order]

    def hybrid_search(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        """Run dense+sparse retrieval and fuse results with Reciprocal Rank Fusion.

        Args:
            query: the user's natural-language / keyword query (EN or AR).
            top_k: number of final results to return.

        Returns:
            A list of the top-k product dicts, best first.

        Raises:
            RuntimeError: if indexes are not built.
            ValueError: if `query` is empty.
        """
        self._require_built()
        query = self._require_query(query)
        if top_k < 1:
            raise ValueError("top_k must be >= 1")

        # vectorised query for the dense side
        dense_query_vec = self._get_model().encode(
            [self._normalize(query)],
            normalize_embeddings=False,
            convert_to_numpy=True,
        )

        # independent top-15 lists from each index
        dense_top15 = self._dense_search_positions(dense_query_vec, 15)
        sparse_top15 = self._sparse_search_positions(self._tokenize(query), 15)

        # RRF fusion — accumulate scores keyed by corpus position
        score_map: Dict[int, float] = {}
        self._rrf_add(score_map, dense_top15)
        self._rrf_add(score_map, sparse_top15)

        # sort by descending RRF score, ties broken by position for stability
        ranked = sorted(score_map.items(), key=lambda kv: (-kv[1], kv[0]))

        return [dict(self.products[pos]) for pos, _score in ranked[:top_k]]

    def get_installment_terms(self) -> Dict[int, float]:
        """Return the store's general installment durations/rates.

        Static policy info (months -> interest rate), independent of any
        specific product. Used to answer "do you offer installments?" style
        questions even when no specific per-product plan has been computed.
        """
        return dict(INSTALMENT_RATES)

    def get_all_categories(self) -> List[str]:
        """Return every distinct category in the catalogue, deterministically.

        Unlike hybrid_search(), this scans the full corpus rather than a
        top-k shortlist, so it's the correct source for "what categories do
        you sell" style questions.
        """
        self._require_built()
        return list(self.categories)

    def keyword_lookup(self, query: str, limit: int = 5) -> List[Dict[str, Any]]:
        """Full-catalogue exact/substring match on title or product_id.

        Safety net for specific product-name or SKU questions: hybrid_search
        only ever considers the top-15 dense + top-15 sparse hits, so an
        exact product mention can still rank outside the final top_k if its
        embedding/BM25 score for THIS phrasing happens to be middling. This
        scans all products directly (cheap at catalogue sizes like this) so
        an exact name/SKU match is never missed just because retrieval
        didn't rank it highly.
        """
        self._require_built()
        norm_query = self._normalize(query)
        if not norm_query:
            return []

        matches: List[int] = []
        for pos, product in enumerate(self.products):
            norm_title = self._normalize(str(product.get("title", "")))
            norm_id = self._normalize(str(product.get("product_id", "")))
            if norm_title and (norm_title in norm_query or norm_query in norm_title):
                matches.append(pos)
            elif norm_id and norm_id and norm_id in norm_query:
                matches.append(pos)
        return [dict(self.products[pos]) for pos in matches[:limit]]

    # ────────────────────────────────────────────────────────────────────
    # C5-C7. Business logic & cross-selling
    # ────────────────────────────────────────────────────────────────────

    def get_recommendations(self, product_id: str, top_k: int = 3) -> List[Dict[str, Any]]:
        """Recommend similar products for cross-selling / up-selling.

        Uses semantic similarity (FAISS top-20) as the candidate pool, then
        enforces a strict ±30% price-band constraint and sorts by rating.

        Args:
            product_id: the id of the product the user is viewing.
            top_k: number of recommendations to return.

        Returns:
            Recommended product dicts (original product excluded).

        Raises:
            RuntimeError: indexes not built.
            LookupError: product_id not found in the corpus.
        """
        self._require_built()

        try:
            origin_pos = next(
                i for i, p in enumerate(self.products) if str(p.get("product_id")) == str(product_id)
            )
        except StopIteration:
            raise LookupError(f"Product id '{product_id}' not found in corpus.")

        original_price = float(self.products[origin_pos].get("final_price", 0.0))
        if original_price <= 0:
            raise ValueError(f"Product '{product_id}' has no valid price.")

        query_vec = np.ascontiguousarray(
            self._get_model().encode(
                [self.rich_texts[origin_pos]],
                normalize_embeddings=False,
                convert_to_numpy=True,
            )
        )
        pool = self._dense_search_positions(query_vec, 20)
        if not pool:
            return []

        # filter out the original and enforce the ±30% price band
        candidates = []
        for pos in pool:
            if pos == origin_pos:
                continue
            price = float(self.products[pos].get("final_price", 0.0))
            if price <= 0:
                continue
            if not (0.7 <= price / original_price <= 1.3):
                continue
            candidates.append((pos, price))

        # sort by rating descending (ties broken by ratings_count desc, then pos)
        candidates.sort(
            key=lambda t: (
                -float(self.products[t[0]].get("rating", 0.0)),
                -float(self.products[t[0]].get("ratings_count", 0.0)),
                t[0],
            )
        )

        return [dict(self.products[pos]) for pos, _price in candidates[:top_k]]

    def calculate_installment(self, price: float, months: int = 6) -> Dict[str, Any]:
        """Compute monthly instalment for a given price.

        Rates: 3 -> 10%, 6 -> 15%, 12 -> 20%.

        Args:
            price: product price.
            months: one of {3, 6, 12}.

        Returns:
            dict with keys: monthly_payment, total_with_interest, months.

        Raises:
            ValueError: on non-positive price or unsupported months.
        """
        if price <= 0:
            raise ValueError("price must be positive.")
        if months not in INSTALMENT_RATES:
            raise ValueError(f"Unsupported instalment months {months!r}; choose from {sorted(INSTALMENT_RATES)}.")

        rate = INSTALMENT_RATES[months]
        total = price * (1.0 + rate)
        monthly = total / months
        return {
            "monthly_payment": round(monthly, 2),
            "total_with_interest": round(total, 2),
            "months": months,
        }

    def should_handoff(self, sentiment_score: float, user_text: str) -> bool:
        """Decide whether to escalate to a human agent.

        Escalates when the VADER sentiment is very negative (< -0.5) OR when
        the user text contains any escalation keyword (English or Arabic).

        Args:
            sentiment_score: VADER-style sentiment in [-1, 1].
            user_text: raw user message.

        Returns:
            True if the conversation should be handed off to a human.
        """
        if sentiment_score < -0.5:
            return True
        lowered = (user_text or "").lower()
        return any(kw in lowered for kw in HANDOFF_KEYWORDS)

    # ────────────────────────────────────────────────────────────────────
    # D8. LLM pipeline context builder
    # ────────────────────────────────────────────────────────────────────

    def build_llm_context(
        self,
        user_message: str,
        intent: str,
        retrieved_products: List[Dict[str, Any]],
        last_5_messages: List[Dict[str, Any]],
        sentiment: float,
    ) -> str:
        """Build a clean, structured prompt context for the external LLM.

        All data is injected as beautifully-formatted JSON (Arabic-safe), so
        the Dual-API manager (OpenRouter / Gemini) and the downstream LLM can
        parse it reliably.

        Args:
            user_message: the latest user utterance.
            intent: classified intent label (e.g. 'product_inquiry').
            retrieved_products: output of hybrid_search (or []).
            last_5_messages: list of {role, content} dicts (chat history).
            sentiment: numeric sentiment in [-1, 1].

        Returns:
            A single formatted string to include in the LLM prompt.
        """

        def _serialize(obj: Any) -> Any:
            """JSON-safe fallback for numpy types / datetimes."""
            if isinstance(obj, (np.integer,)):
                return int(obj)
            if isinstance(obj, (np.floating,)):
                return float(obj)
            if isinstance(obj, np.ndarray):
                return obj.tolist()
            return str(obj)

        context = {
            "user_message": user_message,
            "intent": intent,
            "sentiment": sentiment,
            "conversation_history": last_5_messages[-5:] if last_5_messages else [],
            "retrieved_products": retrieved_products,
        }

        payload = json.dumps(context, indent=2, ensure_ascii=False, default=_serialize)
        return (
            "You are the sales representative for our e-commerce store. "
            "Use ONLY the retrieved product data below to answer, never invent offers.\n\n"
            "=== CONTEXT ===\n"
            f"{payload}\n"
            "=== INSTRUCTIONS ===\n"
            "Formulate a friendly sales pitch: highlight the most relevant product(s), "
            "mention price, and suggest an instalment plan or complementary product when useful."
        )


# ════════════════════════════════════════════════════════════════════════
#  Test block — mock mixed EN/AR dataset and a quick end-to-end run
# ════════════════════════════════════════════════════════════════════════

def _mock_products() -> List[Dict[str, Any]]:
    """Return a small mock catalogue (English + Arabic products)."""
    return [
        {
            "product_id": "ASUS-G15",
            "title": "ASUS ROG Strix G-15 Gaming Laptop",
            "category": "laptops",
            "product_description": "Cheap yet powerful gaming laptop with RTX 4060 and 165Hz display.",
            "product_specifications": "Ryzen 9, 16GB RAM, 1TB SSD, G-15, 1440p",
            "final_price": 1199.00,
            "rating": 4.6,
            "ratings_count": 812,
            "currency": "USD",
        },
        {
            "product_id": "LENOVO-LEGION-5",
            "title": "Lenovo Legion 5 Gaming Laptop",
            "category": "laptops",
            "product_description": "Affordable gaming machine for esports and AAA titles.",
            "product_specifications": "Ryzen 7, 16GB RAM, 512GB SSD, 144Hz",
            "final_price": 1049.00,
            "rating": 4.4,
            "ratings_count": 1240,
            "currency": "USD",
        },
        {
            "product_id": "DELL-XPS-13",
            "title": "Dell XPS 13 Ultrabook",
            "category": "laptops",
            "product_description": "Premium ultra-thin business ultrabook, not for gaming.",
            "product_specifications": "Core i7, 16GB RAM, 512GB SSD",
            "final_price": 1799.00,
            "rating": 4.7,
            "ratings_count": 560,
            "currency": "USD",
        },
        {
            "product_id": "SAMSUNG-MONITOR-27",
            "title": "شاشة سامسونج 27 بوصة للألعاب",
            "category": "monitors",
            "product_description": "شاشة ألعاب بدقة 2K ومعدل تحديث 144 هرتز مع أسعار مناسبة.",
            "product_specifications": "OLED، 2K، 144Hz، G-Sync",
            "final_price": 329.00,
            "rating": 4.3,
            "ratings_count": 431,
            "currency": "USD",
        },
    ]


## 3. LLM Manager \u2014 Dual-API (OpenRouter \u2192 Gemini \u2192 static fallback)

In [18]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import requests


class LLMManager:
    """Dual-API LLM caller: OpenRouter (primary) -> Gemini (backup) -> static fallback."""

    FALLBACK_MESSAGE = (
        "I'm sorry, I'm having temporary technical issues. "
        "Please try again in a moment."
    )

    def __init__(
        self,
        openrouter_model: str = None,
        gemini_model: str = "gemini-2.5-flash",
        temperature: float = 0.2,
        timeout: int = 10,
    ) -> None:
        self.openrouter_model = openrouter_model or os.getenv(
            "OPENROUTER_MODEL", "cohere/north-mini-code:free"
        )
        self.gemini_model = gemini_model
        self.gemini_key = get_secret("GEMINI_API_KEY")
        self.gemini_url = (
            f"https://generativelanguage.googleapis.com/v1beta/models/"
            f"{self.gemini_model}:generateContent"
        )

        # Newer langchain-openai / openai-python versions validate credentials
        # at construction time (not just inside .invoke()), so a missing key
        # would crash the notebook immediately instead of failing gracefully
        # per-call. A placeholder keeps construction safe either way; a real
        # empty/invalid key still fails inside .invoke(), which chat() catches
        # and routes to the Gemini backup / static fallback.
        self.primary_llm = ChatOpenAI(
            model=self.openrouter_model,
            openai_api_key=get_secret("OPEN_ROUTER_KEY") or "not-set",
            openai_api_base="https://openrouter.ai/api/v1",
            temperature=temperature,
            max_retries=1,
            timeout=timeout,
        )

        # Records who answered each turn: "openrouter" | "gemini_backup" | "none"
        self.usage_log: List[Dict[str, str]] = []

    def chat(self, system_prompt: str, user_prompt: str) -> Dict[str, str]:
        """Try OpenRouter, then Gemini, then a static fallback. Never raises."""

        # \u2500\u2500 Attempt 1: OpenRouter (primary) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
        try:
            messages = [SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]
            response = self.primary_llm.invoke(messages)
            self.usage_log.append({"source": "openrouter", "status": "success"})
            return {"content": response.content, "source": "openrouter", "status": "success"}
        except Exception as e:
            print(f"[LLMManager] OpenRouter failed: {e}")

            # \u2500\u2500 Attempt 2: Gemini (backup, free REST API) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
            try:
                text = self._call_gemini(system_prompt, user_prompt)
                self.usage_log.append({"source": "gemini_backup", "status": "fallback"})
                return {"content": text, "source": "gemini_backup", "status": "fallback"}
            except Exception as e2:
                print(f"[LLMManager] Gemini also failed: {e2}")

                # \u2500\u2500 Attempt 3: static fallback \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
                self.usage_log.append({"source": "none", "status": "failed"})
                return {"content": self.FALLBACK_MESSAGE, "source": "none", "status": "failed"}

    def _call_gemini(self, system_prompt: str, user_prompt: str) -> str:
        if not self.gemini_key:
            raise RuntimeError("GEMINI_API_KEY is not set.")
        url = f"{self.gemini_url}?key={self.gemini_key}"
        payload = {
            "contents": [{"parts": [{"text": f"{system_prompt}\n\nUser: {user_prompt}"}]}],
            "generationConfig": {"temperature": 0.2, "maxOutputTokens": 500},
        }
        r = requests.post(url, json=payload, timeout=15)
        r.raise_for_status()
        data = r.json()
        return data["candidates"][0]["content"]["parts"][0]["text"]

## 4. Prompt Builder \u2014 the \"A\" in RAG

In [19]:
class PromptBuilder:
    """Builds the grounded system and user prompt sent to the LLM."""

    SYSTEM_PROMPT = """You are SmartSales Bot, a friendly, professional sales assistant for an e-commerce store.

Rules:
1. Use ONLY the provided product data for product facts, availability, prices, and ratings. Never invent them.
2. If the provided data is insufficient, say "I don't have that information."
3. Keep responses concise, helpful, and professional.
4. Include prices and ratings when mentioning products.
5. When the user describes a support problem, acknowledge it clearly and offer appropriate next steps.
6. Show a supplied installment plan accurately; do not invent plan terms. You may also state the store's general installment durations/rates (given below) even when no specific dollar plan has been computed for a product.
7. Recommend related products only when they are supported by the retrieved product data.
8. Reply in the same language the user wrote in (Arabic or English)."""

    def build_prompt(
        self,
        user_message: str,
        products_found: List[Dict[str, Any]],
        viewed_products: List[Dict[str, Any]],
        chat_history: List[Dict[str, str]],
        installment_info: Optional[Dict[str, Any]] = None,
        all_categories: Optional[List[str]] = None,
        installment_terms: Optional[Dict[int, float]] = None,
    ) -> str:
        return f"""
===============================================
USER MESSAGE: {user_message}
===============================================

POTENTIALLY RELEVANT PRODUCTS FROM SEARCH:
===============================================
{self._format_products(products_found)}

===============================================
FULL CATALOGUE CATEGORY LIST (only when supplied — this is EVERY category
in the store, not just what was retrieved above; use it, and only it, when
the user asks what categories/kinds of products the store carries):
===============================================
{self._format_categories(all_categories)}

===============================================
GENERAL INSTALLMENT TERMS (store policy — always safe to mention, even
without a specific computed plan):
===============================================
{self._format_installment_terms(installment_terms)}

===============================================
INSTALLMENT PLAN COMPUTED FOR A SPECIFIC PRODUCT (only when supplied):
===============================================
{self._format_installment(installment_info)}

===============================================
PRODUCTS USER ALREADY VIEWED:
===============================================
{self._format_viewed(viewed_products)}

===============================================
CONVERSATION HISTORY (last 5 messages):
===============================================
{self._format_history(chat_history)}

===============================================
INSTRUCTIONS:
===============================================
Answer the user's message directly. Use a retrieved product only when it is relevant to the
question. State exact prices and ratings only from the supplied product data. If an installment
plan is supplied, show its terms accurately. If product data does not answer the question, say
what information is unavailable rather than making up a product fact.
"""

    def _format_products(self, products: List[Dict[str, Any]]) -> str:
        if not products:
            return "No products found."
        lines = []
        for i, p in enumerate(products, 1):
            price = p.get("final_price", "N/A")
            initial = p.get("initial_price")
            discount = p.get("discount")
            price_line = f"   Price: ${price}"
            if initial and discount:
                price_line += f" (was ${initial}, {discount}% off)"
            lines.append(
                f"{i}. {p.get('title', 'Unknown product')}\n"
                f"{price_line}\n"
                f"   Rating: {p.get('rating', 'N/A')}/5 ({p.get('ratings_count', 0)} reviews)\n"
                f"   Category: {p.get('category', 'N/A')}\n"
                f"   Customers said: {p.get('what_customers_said', 'N/A')}"
            )
        return "\n\n".join(lines)

    def _format_categories(self, categories: Optional[List[str]]) -> str:
        if not categories:
            return "Not requested for this message."
        return ", ".join(categories)

    def _format_installment_terms(self, terms: Optional[Dict[int, float]]) -> str:
        if not terms:
            return "Not available."
        lines = [f"- {months} months at {rate * 100:.0f}% interest" for months, rate in sorted(terms.items())]
        return "\n".join(lines)

    def _format_installment(self, installment_info: Optional[Dict[str, Any]]) -> str:
        if not installment_info:
            return "No installment plan was supplied."
        return (
            f"{installment_info['months']} months -> "
            f"${installment_info['monthly_payment']}/month "
            f"(total ${installment_info['total_with_interest']})"
        )

    def _format_viewed(self, viewed: List[Dict[str, Any]]) -> str:
        if not viewed:
            return "User hasn't viewed any products yet."
        return ", ".join(p.get("title", "Unknown") for p in viewed)

    def _format_history(self, history: List[Dict[str, str]]) -> str:
        if not history:
            return "This is the start of the conversation."
        lines = []
        for msg in history[-5:]:
            role = "User" if msg.get("role") == "user" else "Bot"
            lines.append(f"{role}: {msg.get('content', '')}")
        return "\n".join(lines)

## 5. Conversation Memory (minimal stand-in for P5)

P5 (*Business Logic & Memory Engineer*) owns the real persistent memory layer per the README. This is a small in-memory placeholder \u2014 just enough state (chat history and last-viewed products, per `session_id`) to exercise `RAGPipeline` end-to-end. Swap it for P5's real class later; `RAGPipeline` only needs the methods below.

In [20]:
class ConversationMemory:
    """In-memory placeholder for P5's persistent memory/session store."""

    def __init__(self) -> None:
        self._sessions: Dict[str, Dict[str, Any]] = {}

    def _session(self, session_id: str) -> Dict[str, Any]:
        return self._sessions.setdefault(session_id, {"messages": [], "viewed_products": []})

    def save_message(self, session_id: str, role: str, content: str) -> None:
        self._session(session_id)["messages"].append({"role": role, "content": content})

    def get_last_n_messages(self, session_id: str, n: int = 5) -> List[Dict[str, str]]:
        return self._session(session_id)["messages"][-n:]

    def get_all_messages(self, session_id: str) -> List[Dict[str, str]]:
        return self._session(session_id)["messages"]

    def add_viewed_products(self, session_id: str, products: List[Dict[str, Any]]) -> None:
        seen_ids = {p.get("product_id") for p in self._session(session_id)["viewed_products"]}
        for p in products:
            if p.get("product_id") not in seen_ids:
                self._session(session_id)["viewed_products"].append(p)
                seen_ids.add(p.get("product_id"))

    def get_viewed_products(self, session_id: str) -> List[Dict[str, Any]]:
        return self._session(session_id)["viewed_products"]

    def get_last_viewed_product(self, session_id: str) -> Optional[Dict[str, Any]]:
        viewed = self._session(session_id)["viewed_products"]
        return viewed[-1] if viewed else None

## 6. Explicit human-handoff policy

Human handoff no longer depends on any classifier or score. `HumanHandoffPolicy` has two independent routes:

1. The frontend can set `request_handoff=True`, for example from a **Contact a representative** control.
2. The customer directly asks to speak with a person using the supported English or Arabic phrases.

This keeps escalation deterministic and auditable. It avoids inferring whether a customer should be transferred from their language or tone. The policy returns a reason code for the frontend and for the agent summary.

In [21]:
class HumanHandoffPolicy:
    """Route explicit requests for a human representative without model scoring."""

    _EXPLICIT_REQUEST_PHRASES = (
        "talk to a human",
        "speak to a human",
        "talk to a person",
        "speak to a person",
        "human agent",
        "live agent",
        "human representative",
        "customer representative",
        "contact support",
        "customer service",
        "\u0645\u0648\u0638\u0641 \u062e\u062f\u0645\u0629 \u0627\u0644\u0639\u0645\u0644\u0627\u0621",
        "\u062f\u0639\u0645 \u0628\u0634\u0631\u064a",
        "\u0627\u062a\u0643\u0644\u0645 \u0645\u0639 \u062d\u062f",
        "\u0627\u062a\u0643\u0644\u0645 \u0645\u0639 \u0634\u062e\u0635",
        "\u0627\u0643\u0644\u0645 \u062d\u062f",
        "\u0623\u0643\u0644\u0645 \u062d\u062f",
        "\u0623\u062a\u0643\u0644\u0645 \u0645\u0639 \u062d\u062f",
        "\u0645\u0648\u0638\u0641",
    )

    def evaluate(self, user_message: str, request_handoff: bool = False) -> Optional[str]:
        """Return a reason code when an explicit handoff route is requested."""
        if request_handoff:
            return "frontend_request"

        normalized = " ".join((user_message or "").casefold().split())
        if any(phrase in normalized for phrase in self._EXPLICIT_REQUEST_PHRASES):
            return "explicit_customer_request"
        return None

## 7. RAG Pipeline \u2014 wiring P2 and P4 together

`process_message()` is the single entry point that a Gradio `app.py` would call for each user turn. It:

1. Applies the deterministic `HumanHandoffPolicy` before generating a reply.
2. Retrieves potentially relevant products through `engine.hybrid_search()`.
3. Optionally calculates a plan only when the frontend explicitly supplies `installment_months`.
4. Builds the grounded prompt and calls the LLM.
5. Saves the turn to memory and returns a structured result.

In [22]:
class RAGPipeline:
    """Orchestrates P2 retrieval and P4 prompt generation, memory, handoff, and LLM calls."""

    # Matches "aggregate / whole-catalogue" questions that a top-k retrieval
    # shortlist structurally cannot answer completely or deterministically
    # (e.g. "what categories do you have", "list your categories").
    _CATEGORY_INTENT_RE = re.compile(
        r"(what|which).{0,20}\bcategor(y|ies)\b"
        r"|\blist\b.{0,20}\bcategor(y|ies)\b"
        r"|\bcategories\b.{0,20}\b(you|do you|available)\b"
        r"|\bwhat.{0,15}(kinds?|types?).{0,10}(of )?products?\b"
        r"|\u0627\u0644\u0641\u0626\u0627\u062a\b|\u0627\u0644\u0623\u0642\u0633\u0627\u0645\b"
        r"|\u0623\u0646\u0648\u0627\u0639\s+\u0627\u0644\u0645\u0646\u062a\u062c\u0627\u062a",
        re.IGNORECASE,
    )

    def __init__(
        self,
        engine: "SalesRetrievalEngine",
        llm_manager: Optional[LLMManager] = None,
        prompt_builder: Optional[PromptBuilder] = None,
        memory: Optional[ConversationMemory] = None,
        handoff_policy: Optional[HumanHandoffPolicy] = None,
    ) -> None:
        self.engine = engine
        self.llm = llm_manager or LLMManager()
        self.prompt_builder = prompt_builder or PromptBuilder()
        self.memory = memory or ConversationMemory()
        self.handoff_policy = handoff_policy or HumanHandoffPolicy()

    def process_message(
        self,
        user_message: str,
        session_id: str,
        request_handoff: bool = False,
        installment_months: Optional[int] = None,
    ) -> Dict[str, Any]:
        """Handle one message without classifier-derived fields or dependencies.

        `request_handoff` is a frontend-controlled explicit action. `installment_months` is also
        frontend-controlled, allowing a UI to request a calculation for the customer's last viewed
        product without interpreting the customer's message in this pipeline.
        """
        handoff_reason = self.handoff_policy.evaluate(user_message, request_handoff)
        self.memory.save_message(session_id, "user", user_message)

        if handoff_reason:
            summary = self._generate_handoff_summary(session_id, handoff_reason)
            return {
                "type": "handoff",
                "message": "I\u2019ll connect you with a human representative.",
                "summary_for_agent": summary,
                "handoff_reason": handoff_reason,
            }

        products = self._retrieve_products(user_message)
        installment_info = self._build_installment_plan(session_id, installment_months)
        installment_terms = self.engine.get_installment_terms()
        all_categories = (
            self.engine.get_all_categories()
            if self._CATEGORY_INTENT_RE.search(user_message or "")
            else None
        )

        chat_history = self.memory.get_last_n_messages(session_id, n=5)
        viewed_products = self.memory.get_viewed_products(session_id)
        prompt = self.prompt_builder.build_prompt(
            user_message=user_message,
            products_found=products,
            viewed_products=viewed_products,
            chat_history=chat_history,
            installment_info=installment_info,
            all_categories=all_categories,
            installment_terms=installment_terms,
        )

        llm_response = self.llm.chat(
            system_prompt=self.prompt_builder.SYSTEM_PROMPT,
            user_prompt=prompt,
        )

        self.memory.save_message(session_id, "bot", llm_response["content"])
        if products:
            self.memory.add_viewed_products(session_id, products[:1])

        return {
            "type": "response",
            "message": llm_response["content"],
            "products": products,
            "installment": installment_info,
            "llm_source": llm_response["source"],
        }

    def _retrieve_products(self, user_message: str) -> List[Dict[str, Any]]:
        """Retrieve evidence for every non-empty message; an empty message has no search query.

        Combines the semantic/BM25 top-k shortlist with a full-catalogue
        exact/substring lookup, so a product named or SKU-referenced
        explicitly isn't dropped just because its embedding/BM25 score for
        this exact phrasing put it outside the top-5.
        """
        text = (user_message or "").strip()
        if not text:
            return []
        try:
            results = self.engine.hybrid_search(text, top_k=5)
        except Exception as error:
            print(f"[RAGPipeline] Retrieval failed: {error}")
            results = []

        try:
            exact_matches = self.engine.keyword_lookup(text, limit=3)
        except Exception as error:
            print(f"[RAGPipeline] Keyword lookup failed: {error}")
            exact_matches = []

        seen_ids = {p.get("product_id") for p in results}
        for product in exact_matches:
            if product.get("product_id") not in seen_ids:
                results.append(product)
                seen_ids.add(product.get("product_id"))
        return results

    def _build_installment_plan(
        self, session_id: str, installment_months: Optional[int]
    ) -> Optional[Dict[str, Any]]:
        """Calculate a plan only when the frontend explicitly supplies valid month count."""
        if (
            not isinstance(installment_months, int)
            or isinstance(installment_months, bool)
            or installment_months <= 0
        ):
            return None

        viewed = self.memory.get_last_viewed_product(session_id)
        if not viewed or not viewed.get("final_price"):
            return None
        return self.engine.calculate_installment(
            float(viewed["final_price"]), months=installment_months
        )

    def _generate_handoff_summary(self, session_id: str, handoff_reason: str) -> str:
        history = self.memory.get_all_messages(session_id)
        summary_prompt = (
            "Summarize this customer conversation for a human agent. "
            "Include the latest request, products viewed, any unresolved support issue, and the "
            f"handoff reason code ({handoff_reason}). Be concise and factual.\n\n"
            f"Conversation:\n{history}"
        )
        result = self.llm.chat(
            "You are a conversation summarizer. Be concise and factual.",
            summary_prompt,
        )
        return result["content"]

## 8. Build the engine from the real catalogue & demo run

Builds the FAISS + BM25 indexes from the **913-product `products_clean.csv` catalogue** (fashion/accessories: backpacks, shirts, dresses, jeans, sports shoes, etc.) instead of the old 4-item mock catalogue.

If `OPENROUTER_API_KEY` and `GEMINI_API_KEY` are not set, `llm_source` will show `\"none\"` and `message` will be the static fallback. That is expected offline; retrieval, prompt, memory, and explicit handoff wiring still run normally.

In [25]:
products = load_products_from_csv()

engine = SalesRetrievalEngine()
engine.build_indexes(products)

pipeline = RAGPipeline(engine)
session_id = "demo-session-1"

conversation = [
    {"message": "Hi, what is the best headphones i can buy?"},
    {"message": "Is there a cheaper headphones options?"},
    {"message": "Can I pay in installments?"},
    {"message": "Can you calculate what i will pay in 6 months for the BLAUPUNKT headphone?"},
]

for turn in conversation:
    result = pipeline.process_message(
        turn["message"],
        session_id=session_id,
        request_handoff=turn.get("request_handoff", False),
        installment_months=turn.get("installment_months"),
    )
    print("=" * 70)
    print("USER:", turn["message"])
    print("-" * 70)
    print(
        f"type={result['type']}  "
        f"handoff_reason={result.get('handoff_reason', 'n/a')}  "
        f"llm_source={result.get('llm_source', 'n/a')}"
    )
    print("-" * 70)
    print("BOT:", result["message"])
    if result.get("products"):
        print("\nProducts surfaced:", [p["title"] for p in result["products"]])
    if result.get("installment"):
        print("Installment plan:", result["installment"])
    if result.get("summary_for_agent"):
        print("\nHandoff summary for agent:", result["summary_for_agent"])
    print()

Loaded 913 active products from 'products_clean.csv'


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

USER: Hi, what is the best headphones i can buy?
----------------------------------------------------------------------
type=response  handoff_reason=n/a  llm_source=openrouter
----------------------------------------------------------------------
BOT: Based on the available data, the **BLAUPUNKT headphones** appear to be the best choice:

- **Price:** $47.99 (originally $71.99, 33 % off)
- **Rating:** 4.5 / 5 stars (11 reviews)

They have the highest rating among the headphones listed and a competitive price. If you’d like to spread the cost, our store offers installment plans such as:

- 3 months at 10 % interest  
- 6 months at 15 % interest  
- 12 months at 20 % interest  

Let me know if you’d like more details or assistance with the purchase!

Products surfaced: ['CrossBeats', 'BLAUPUNKT', 'NOISE', 'Earth Rhythm', 'Cutiekins', 'W', 'W', 'W']

USER: Is there a cheaper headphones options?
----------------------------------------------------------------------
type=response  handoff_

In [24]:
# LLM usage log \u2014 which API answered each turn (useful for an ops/admin dashboard)
pipeline.llm.usage_log

[{'source': 'openrouter', 'status': 'success'},
 {'source': 'openrouter', 'status': 'success'},
 {'source': 'openrouter', 'status': 'success'},
 {'source': 'openrouter', 'status': 'success'},
 {'source': 'openrouter', 'status': 'success'}]